# Reliable RAG

Add guard rails: grade retrieved chunks, generate, then verify the answer is grounded.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

## Step 1: grade every retrieved chunk
Only chunks that the LLM judges relevant go forward.

In [ ]:
def is_relevant(query, chunk):
    return llm_yes(f"Question: {query}\n\nPassage: {chunk}\n\nDoes the passage contain information useful for answering the question?")

def grounded(answer_text, contexts):
    ctx = "\n".join(contexts)
    return llm_yes(f"Context:\n{ctx}\n\nAnswer: {answer_text}\n\nIs every claim in the answer supported by the context?")

## Step 2: the reliable pipeline
If nothing relevant is found, or the answer is not grounded, we refuse instead of hallucinating.

In [ ]:
def reliable_rag(query, k=6, max_retries=1):
    hits = store.search(query, k)
    good = [h["text"] for h in hits if is_relevant(query, h["text"])]
    if not good:
        return "I could not find relevant information.", []
    for _ in range(max_retries + 1):
        response = answer(query, good)
        if grounded(response, good):
            return response, good
    return "I could not produce a well-grounded answer.", good

for q in ["What is Reciprocal Rank Fusion?", "What is the capital of Australia?"]:
    print("Q:", q, "\nA:", reliable_rag(q)[0], "\n")